# Settings

In [1]:
library(ggplot2)
library(patchwork)
library(glmGamPoi)
library(gplots)
library(RColorBrewer)
library(grid)
library(gridExtra)
library(cowplot)
library(dplyr)
library(hrbrthemes)
library(viridis)
library(preprocessCore)
library(fields)
library(ExPosition)
library(tidyr)
library(dplyr)
library(reshape2)
library(ggridges)
set.seed(1234)

Warning message:
“package ‘glmGamPoi’ was built under R version 4.4.2”

Attaching package: ‘glmGamPoi’


The following object is masked from ‘package:ggplot2’:

    vars



Attaching package: ‘gplots’


The following object is masked from ‘package:stats’:

    lowess



Attaching package: ‘cowplot’


The following object is masked from ‘package:patchwork’:

    align_plots



Attaching package: ‘dplyr’


The following object is masked from ‘package:gridExtra’:

    combine


The following object is masked from ‘package:glmGamPoi’:

    vars


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: viridisLite

Warning message:
“package ‘preprocessCore’ was built under R version 4.4.2”
Loading required package: spam

Spam version 2.10-0 (2023-10-23) is loaded.
Type 'help( Spam)' or 'demo( spam)' for a short introduction 
and overview of this packag

In [2]:
# input files
rowdata <- readRDS("../format_expression_data/rowdata_mouse.rds")
coldata <- readRDS("../format_expression_data/ENCODE/coldata.rds")
exprmat <- readRDS("../format_expression_data/ENCODE/exprmat.rds")
layout <- readRDS("../format_expression_data/ENCODE/layout.rds")
out.file.prefix <- "encode_"

In [3]:
color.pal <- list("CL"="#cc79a3",
                  "DL"="#d55e00",
                  "SV"="#e69f00",
                  "VP"="#56b4e9",
                  "VnP"="#0072b2")
rowdata$fusil <- factor(rowdata$fusil, levels=names(color.pal))

# Filter unmeasured genes

In [4]:
print(paste0("N genes exprmat: ", nrow(exprmat)))
print(paste0("N genes rowdata: ", nrow(rowdata)))

any.na <- function(vals){
    return(any(is.na(vals)))}
temp.keep <- which(!apply(exprmat, MARGIN=1, FUN=any.na))
exprmat.filt <- exprmat[temp.keep,]
rowdata.filt <- rowdata[temp.keep,]
print(paste("N genes after filt NAs exprmat: ", nrow(exprmat.filt), sep=""))
print(paste("N genes after filt NAs rowdata: ", nrow(rowdata.filt), sep=""))

all.zero <- function(vals){
    return(all(vals==0))}
temp.keep <- which(!apply(exprmat.filt, MARGIN=1, FUN=all.zero))
exprmat.filt <- exprmat.filt[temp.keep,]
rowdata.filt <- rowdata.filt[temp.keep,]
print(paste("N genes after filt 0s exprmat: ", nrow(exprmat.filt), sep=""))
print(paste("N genes after filt 0s rowsata: ", nrow(rowdata.filt), sep=""))

[1] "N genes exprmat: 42319"
[1] "N genes rowdata: 42319"
[1] "N genes after filt NAs exprmat: 37913"
[1] "N genes after filt NAs rowdata: 37913"
[1] "N genes after filt 0s exprmat: 33957"
[1] "N genes after filt 0s rowsata: 33957"


# explore expression distributions

In [ ]:
exprmat.filt.norm <- normalize.quantiles(exprmat.filt)
rownames(exprmat.filt.norm) <- rownames(exprmat.filt)
colnames(exprmat.filt.norm) <- colnames(exprmat.filt)

p <- exprmat.filt %>% 
    melt() %>%
    'colnames<-'(c("mgi_id", "sample", "tpm")) %>%
    mutate(tpm_log=log2(tpm+1)) %>%
    ggplot(aes(x=tpm_log, y=sample)) + geom_violin() + 
    stat_summary(fun=median, geom="point", size=2) + theme_classic()

p.norm <- exprmat.filt.norm %>% 
    melt() %>%
    'colnames<-'(c("mgi_id", "sample", "tpm")) %>%
    mutate(tpm_log=log2(tpm+1)) %>%
    ggplot(aes(x=tpm_log, y=sample)) + geom_violin() + 
    stat_summary(fun=median, geom="point", size=2) + theme_classic()

pdf(paste0(out.file.prefix,"expression_dist.pdf"))
p
p.norm
dev.off()

p
p.norm

pdf 
  2

# Plot expression in each sample by FUSIL category

In [ ]:
plot.list <- list()
for(i in seq(nrow(coldata))){
    plot.list[[i]] <- exprmat.filt %>% 
        melt() %>%
        'colnames<-'(c("mgi_id", "sample", "tpm")) %>%
        filter(sample==rownames(coldata)[i]) %>% 
        mutate(tpm_log=log2(tpm+1)) %>% 
        left_join(tibble::rownames_to_column(rowdata, var="mgi_id") %>% select(c("mgi_id", "fusil")), by="mgi_id") %>% 
        filter(!is.na(fusil)) %>% 
        ggplot(aes(x=fusil, y=tpm_log, fill=fusil)) + geom_violin() + geom_boxplot(width=0.15, fill="white") +
        theme_classic() + theme(legend.position = "none") + scale_fill_manual(values=unlist(color.pal)) + 
        ggtitle(rownames(coldata)[i]) + theme(text=element_text(size=20)) +
        xlab("FUSIL Category") + ylab("log2(TPM+1)")
}

pdf(paste0(out.file.prefix,"expression_per_sample_per_fusil.pdf"))
for(i in seq(nrow(coldata))){
    print(plot.list[[i]])}
dev.off()

plot.list[[1]]
plot.list[[5]]

# Calculate and plot tau

In [ ]:
# function to calculate tau as in https://pubmed.ncbi.nlm.nih.gov/15388519/ & https://pubmed.ncbi.nlm.nih.gov/26891983/
calc.tau <- function(vals){
    return(sum(1-vals/max(vals))/(length(vals)-1))
}

p <- apply(exprmat.filt.norm, 1, calc.tau) %>%
    data.frame() %>% 
    tibble::rownames_to_column() %>%
    'colnames<-'(c("mgi_id", "tau")) %>%
    left_join(tibble::rownames_to_column(rowdata, var="mgi_id") %>% select(c("mgi_id", "fusil")), by="mgi_id") %>% 
    filter(!is.na(fusil)) %>% 
    mutate(fusil=factor(fusil, levels=rev(names(color.pal)))) %>%
    ggplot(aes(x=tau, y=fusil, fill=fusil)) + geom_density_ridges() +
    theme_classic() + theme(legend.position = "none") + scale_fill_manual(values=unlist(color.pal)) + 
    theme(text=element_text(size=20)) +
    xlab("Tau") + ylab("FUSIL category") + scale_x_continuous(limits=c(-0,1.1), breaks=c(0,1))

pdf(paste0(out.file.prefix,"tau.pdf"))
p
dev.off()

p